# Magnetfeltet fra en spole med Biot–Savarts lov

Formelen B = μ₀NI/l for feltet inne i en spole gjelder bare for en lang spole, langt fra endene. Hva er feltet ved enden, eller i en kort spole? Med Biot–Savarts lov kan vi regne ut feltet fra en liten bit av ledningen om gangen, og la Python legge sammen alle bitene.

Oppgavene står på nettsiden: [Magnetfeltet fra en spole med Biot–Savarts lov](https://fysikklab.no/programmering/python-biot-savart.html). Kjør cellene i rekkefølge med **Shift + Enter**.

> **Lagring:** Endringene dine lagres bare i denne nettleseren. Vil du ta vare på arbeidet eller levere det, velger du *File → Download*.

## Steg 1: Én strømsløyfe

Biot–Savarts lov sier at en liten bit dl av en ledning med strømmen I gir feltet dB = μ₀I/(4π) · dl × r / r³, der r er vektoren fra biten til punktet der vi vil vite feltet. Vi deler en sirkelformet sløyfe i 200 biter og summerer. Kryssproduktet gjør `np.cross`. I sentrum av en sløyfe er B = μ₀I/(2R).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

mu0 = 4 * np.pi * 1e-7     # T m/A

def ring(R, z0, n=200):
    """Deler en strømsløyfe med radius R i planet z = z0 i n små biter.
    Gir midtpunktet og lengdevektoren dl til hver bit."""
    fi = np.linspace(0, 2 * np.pi, n + 1)
    punkter = np.column_stack([R * np.cos(fi), R * np.sin(fi), np.full(n + 1, z0)])
    dl = punkter[1:] - punkter[:-1]
    midt = (punkter[1:] + punkter[:-1]) / 2
    return midt, dl

def felt(biter, I, punkt):
    """Biot–Savarts lov: dB = μ0 I / (4π) · dl × r / r³, summert over alle bitene."""
    midt, dl = biter
    r = punkt - midt                              # fra hver bit til punktet
    r3 = np.linalg.norm(r, axis=1)**3
    dB = mu0 * I / (4 * np.pi) * np.cross(dl, r) / r3[:, None]
    return dB.sum(axis=0)                         # (Bx, By, Bz)

R, I = 0.10, 5.0        # m, A
B = felt(ring(R, 0), I, np.array([0, 0, 0]))
print(f"B i sentrum: {B[2] * 1e6:.3f} µT (numerisk)")
print(f"μ0 I / (2R):  {mu0 * I / (2 * R) * 1e6:.3f} µT (formel)")

## Steg 2: Feltet langs aksen

Vi regner ut feltet i 121 punkter på aksen gjennom sløyfa, og sammenligner med formelen B = μ₀IR² / (2(R² + z²)<sup>3/2</sup>), som gjelder på aksen. Koden bruker funksjonene fra steg 1.

In [ ]:
# Feltet langs aksen gjennom sløyfa
z = np.linspace(-0.3, 0.3, 121)
biter = ring(R, 0)
Bz = [felt(biter, I, np.array([0, 0, zi]))[2] for zi in z]
B_formel = mu0 * I * R**2 / (2 * (R**2 + z**2)**1.5)

plt.plot(z * 100, np.array(Bz) * 1e6, label="Biot–Savart, numerisk")
plt.plot(z * 100, B_formel * 1e6, "--", label="formel på aksen")
plt.xlabel("z / cm")
plt.ylabel("$B_z$ / µT")
plt.legend()
plt.grid(True)
plt.show()

## Steg 3: En spole

En spole er mange sløyfer etter hverandre. Spolen har 200 vindinger på 40 cm og radius 2 cm. Det blir 20 000 biter, og feltet regnes ut i 161 punkter. Det tar litt tid.

In [ ]:
# En spole: N vindinger fordelt jevnt over lengden l
def spole(N, R, l, n=100):
    midt, dl = [], []
    for z0 in np.linspace(-l / 2, l / 2, N):
        m, d = ring(R, z0, n)
        midt.append(m)
        dl.append(d)
    return np.vstack(midt), np.vstack(dl)

N, l = 200, 0.40          # vindinger og lengde i m
R = 0.02                  # m
biter = spole(N, R, l)
z = np.linspace(-0.4, 0.4, 161)
Bz = np.array([felt(biter, I, np.array([0, 0, zi]))[2] for zi in z])

B_lang = mu0 * N * I / l
print(f"Midt i spolen: {Bz[80] * 1000:.3f} mT (numerisk)")
print(f"μ0 N I / l:     {B_lang * 1000:.3f} mT (lang spole)")
print(f"Ved enden:     {Bz[np.argmin(abs(z - l / 2))] * 1000:.3f} mT")

plt.plot(z * 100, Bz * 1000)
plt.axhline(B_lang * 1000, color="gray", linestyle="--", label="μ0NI/l")
plt.axvspan(-l / 2 * 100, l / 2 * 100, color="gray", alpha=0.15, label="spolen")
plt.xlabel("z / cm")
plt.ylabel("$B_z$ / mT")
plt.legend()
plt.grid(True)
plt.show()

## Din egen kode

Bruk cellen under til oppgavene. Du kan lage flere celler med **+** i verktøylinja.